# Guessing transition states by interpolating force fields

This notebook demonstrates how to use the `TransitionStateGuesser` class to generate initial guesses for transition state optimisation. This is applied to the hydrolysis of ethyl acetate, for which the transition state is then optimised. Generating the initial guess takes between 5 and 15 minutes depending on the computational resources and options. Because of that, data files with the results are provided in the repository that can be readily visualised

## Generating the transition state guess

The first step is to import veloxchem. Information on how to install veloxchem can be found at https://veloxchem.org/docs/installation.html#installing-using-conda

In [ ]:
import veloxchem as vlx

The first step is to define the molecules. This can be done in several ways, for example by SMILES strings:

In [ ]:
ethyl_acetate = vlx.Molecule.read_smiles('CC(=O)OCC')
water = vlx.Molecule.read_smiles('O')
acetic_acid = vlx.Molecule.read_smiles('CC(=O)O')
ethanol = vlx.Molecule.read_smiles('OCC')

print("Ethyl acetate", flush=True)
ethyl_acetate.show(height=200)
print("Water", flush=True)
water.show(height=200)
print("Acetic acid", flush=True)
acetic_acid.show(height=200)
print("Ethanol", flush=True)
ethanol.show(height=200)

With the molecules created, the transition state guesser can be called to take care of everything. A couple possible options are displayed that can be enabled if desired.

**Note**: To immediatly visualise pre-computed results included with the repository, skip two cells ahead

In [ ]:
tsguesser = vlx.TransitionStateGuesser()

# Optional settings
tsguesser.do_qm_scan = False  # Calculates the QM energy of every configuration in the scan (Default: False)
tsguesser.peak_conformer_search = True  # Performs conformer search for some lambda values around the peak structure (Default: False)
tsguesser.force_conformer_search = False  # Performs conformer search for all structures in the scan (Default: False)
tsguesser.mm_scan_backward = False  # Performs the MM scan from products to reactants as well (Default: False)

# Override the default filename
tsguesser.results_file = 'ester_hydrolysis_results_new.h5'

results = tsguesser.find_transition_state(
    [ethyl_acetate, water],
    [acetic_acid, ethanol],
)

And these results can then be visualised as follows

In [ ]:
tsguesser.show_results(results)

Alternatively, pre-calculated results can be loaded from an H5 file:

In [ ]:
vlx.TransitionStateGuesser()
results = tsguesser.load_results('ester_hydrolysis_results.h5')
tsguesser.show_results(results)

The results dictionary has dedicated entries for the best guess based on MM and QM energies.

In [ ]:
print(results['max_mm_xyz'])
print(results['max_qm_xyz'])  # Only defined if scf_scan was set to true

Alternatively, any desired structure can be indexed as follows

In [ ]:
results['scan'][0.5][0]

## Optimising the transition state

And these structures can then be used to optimise a transition state

In [ ]:
# Create a Molecule object from the XYZ string of the highest energy structure
molecule = vlx.Molecule.read_xyz_string(results['max_qm_xyz'])

# Define the basis set
basis = vlx.MolecularBasis.read(molecule, 'Def2-SVP')

# Calculate the SCF energy
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.ostream.mute()
scf_drv.xcfun = "PBE0"
scf_results = scf_drv.compute(molecule, basis)

# Using the SCF results, start the transition state optimisation
opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.transition = True
opt_results = opt_drv.compute(molecule, basis, scf_results)

And the results can be visualised as follows

In [ ]:
opt_drv.show_convergence(opt_results)

To confirm this is indeed a transition state, a vibrational analysis can be performed

In [ ]:
transition_state = vlx.Molecule.read_xyz_string(opt_results['final_geometry'])
vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_results = vib_drv.compute(transition_state, basis)

For which the results can be printed and visualised as follows

In [ ]:
vib_drv.print_info(vib_results)
vib_drv.animate(vib_results, mode=1)

## Forcing breaking and forming bonds

Lastly, bonds can be forcibly broken or formed to create more complex reactions

In [ ]:
tsguesser = vlx.TransitionStateGuesser()

# Optional settings
tsguesser.mute_ff_build = False
tsguesser.results_file = 'water_assisted_ester_hydrolysis_results_new.h5'
tsguesser.do_qm_scan = False
tsguesser.mm_scan_backward = False
tsguesser.peak_conformer_search = True

results = tsguesser.find_transition_state(
    [water, water, ethyl_acetate],
    [water, acetic_acid, ethanol],
    forced_breaking_bonds=[(1, 2), (4, 5)],
)


And these results can be visualised as well

In [ ]:
tsguesser = vlx.TransitionStateGuesser()
results = tsguesser.load_results(
    'water_assisted_ester_hydrolysis_results_new.h5')
tsguesser.show_results(results)